In [0]:
# ============================================
# CAMADA BRONZE - Salvar como Delta Tables
# ============================================
from pyspark.sql.functions import current_timestamp, lit

BUCKET_PATH = "s3://data-lake-portfolio-priscila-2026"
BRONZE_PATH = f"{BUCKET_PATH}/bronze"

def ingest_bronze(df, nome_tabela):
    """Salva dados brutos como Delta Table com metadata de ingestão."""
    df_com_metadata = df \
        .withColumn("_ingested_at", current_timestamp()) \
        .withColumn("_source", lit("s3_raw"))
    
    df_com_metadata.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{BRONZE_PATH}/{nome_tabela}")
    
    count = df_com_metadata.count()
    print(f"✅ Bronze/{nome_tabela}: {count} linhas salvas")
    return df_com_metadata

# Lendo raw
df_clientes  = spark.read.csv(f"{BUCKET_PATH}/clientes.csv",  header=True, inferSchema=True)
df_transacoes = spark.read.csv(f"{BUCKET_PATH}/transacoes.csv", header=True, inferSchema=True)
df_logs      = spark.read.csv(f"{BUCKET_PATH}/logs_seguranca.csv", header=True, inferSchema=True)

# Salvando Bronze
ingest_bronze(df_clientes,  "clientes")
ingest_bronze(df_transacoes, "transacoes")
ingest_bronze(df_logs,      "logs_seguranca")

print("\n🥉 Camada Bronze concluída!")